# Digit Recognition with PyTorch
We classify handwritten digits from the [MNIST](http://yann.lecun.com/exdb/mnist/) dataset with a fully connected network. The exercise cells marked **TODO** are yours to fill in; `Digit_Recognition_PyTorch_Solution.ipynb` has a complete version.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

# Use the GPU when there is one; otherwise fall back to the CPU.
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print("Using device:", device)

## Retrieve the data
`torchvision` downloads MNIST the first time and caches it in `data/`. `ToTensor` turns each 28x28 image into a tensor with values in 0-1.

In [ ]:
transform = transforms.ToTensor()
train_set = datasets.MNIST('data', train=True, download=True, transform=transform)
test_set = datasets.MNIST('data', train=False, download=True, transform=transform)

print(len(train_set), "training images,", len(test_set), "test images")

## Visualize the training data
Each image has a label from 0 to 9.

In [ ]:
def display_digit(index):
    image, label = train_set[index]
    plt.title('Training data, index: %d,  Label: %d' % (index, label))
    plt.imshow(image.squeeze(0), cmap='gray_r')
    plt.show()

display_digit(0)

## Building the network
**TODO:** define `model` as an `nn.Sequential` that takes a batch of images, flattens each into 784 values (`nn.Flatten()`), passes them through hidden layers with a `ReLU` activation, and outputs 10 scores, one per digit. Don't add a softmax: `nn.CrossEntropyLoss` includes it.

Then create the loss and an optimizer (try `torch.optim.Adam(model.parameters(), lr=0.001)`).

In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Move the model to the GPU
If a GPU is available, this copies the model's weights to it (on a CPU-only machine it does nothing). Do this before training, and remember that every batch of data must be on the same device as the model.

In [ ]:
model = model.to(device)

## Training the network
**TODO:** write the training loop. For every mini-batch: zero the gradients, compute the loss, call `loss.backward()`, and take an optimizer step.

In [ ]:
train_loader = DataLoader(train_set, batch_size=100, shuffle=True)
test_loader = DataLoader(test_set, batch_size=1000)

epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print("Epoch %d  mean training loss %.4f" % (epoch + 1, running_loss / len(train_loader)))

## Testing
How well does the trained network do on images it has never seen? We aim for better than 95%.

In [ ]:
model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        correct += (model(images).argmax(dim=1) == labels).sum().item()

test_accuracy = correct / len(test_set)
print("Test accuracy: %.4f" % test_accuracy)
assert test_accuracy > 0.95, "Try a bigger network or more epochs"